# Preliminary round 01: data merge, `is_late` labelling, EDA and a CatBoost baseline

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import catboost as cb
import lightgbm as lgb
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Load Data

In [ ]:
train_orders_df = pd.read_csv("dataset/train/df_Orders.csv")
train_customers_df = pd.read_csv("dataset/train/df_Customers.csv").drop("customer_id", axis=1)
train_ordersitems_df = pd.read_csv("dataset/train/df_OrderItems.csv").drop("order_id", axis=1)
train_payments_df = pd.read_csv("dataset/train/df_Payments.csv").drop("order_id", axis=1)
train_products_df = pd.read_csv("dataset/train/df_Products.csv").drop("product_id", axis=1)
ori_train_df = pd.concat(
    [
        train_orders_df,
        train_customers_df,
        train_ordersitems_df,
        train_payments_df,
        train_products_df,
    ],
    axis=1,
)
ori_train_df["order_delivered_timestamp"] = pd.to_datetime(ori_train_df["order_delivered_timestamp"])
ori_train_df["order_purchase_timestamp"] = pd.to_datetime(ori_train_df["order_purchase_timestamp"])
ori_train_df["order_approved_at"] = pd.to_datetime(ori_train_df["order_approved_at"])
ori_train_df["order_estimated_delivery_date"] = pd.to_datetime(ori_train_df["order_estimated_delivery_date"])

test_orders_df = pd.read_csv("dataset/test/df_Orders.csv")
test_customers_df = pd.read_csv("dataset/test/df_Customers.csv").drop("customer_id", axis=1)
test_ordersitems_df = pd.read_csv("dataset/test/df_OrderItems.csv").drop("order_id", axis=1)
test_payments_df = pd.read_csv("dataset/test/df_Payments.csv").drop("order_id", axis=1)
test_products_df = pd.read_csv("dataset/test/df_Products.csv").drop("product_id", axis=1)
ori_test_df = pd.concat(
    [
        test_orders_df,
        test_customers_df,
        test_ordersitems_df,
        test_payments_df,
        test_products_df,
    ],
    axis=1,
)
ori_test_df["order_purchase_timestamp"] = pd.to_datetime(ori_test_df["order_purchase_timestamp"])
ori_test_df["order_approved_at"] = pd.to_datetime(ori_test_df["order_approved_at"])

ori_train_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,customer_state,...,shipping_charges,payment_sequential,payment_type,payment_installments,payment_value,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,Axfy13Hk4PIk,hCT0x9JiGXBQ,delivered,2017-10-22 18:57:54,2017-10-22 19:14:13,2017-10-26 22:19:52,2017-11-09,58125,varzea paulista,SP,...,84.65,1,credit_card,1,259.14,toys,491.0,19.0,12.0,16.0
1,v6px92oS8cLG,PxA7fv9spyhx,delivered,2018-06-20 21:40:31,2018-06-20 22:20:20,2018-07-03 22:51:22,2018-07-24,3112,armacao dos buzios,RJ,...,23.79,1,credit_card,8,382.39,watches_gifts,440.0,18.0,14.0,17.0
2,Ulpf9skrhjfm,g3nXeJkGI0Qw,delivered,2018-02-16 16:19:31,2018-02-17 16:15:35,2018-02-27 01:29:50,2018-03-08,4119,jandira,SP,...,17.38,1,credit_card,4,249.25,costruction_tools_garden,2200.0,16.0,16.0,16.0
3,bwJVWupf2keN,EOEsCQ6QlpIg,delivered,2018-08-18 18:04:29,2018-08-18 18:15:16,2018-08-27 20:03:51,2018-09-19,18212,uberlandia,MG,...,30.72,1,credit_card,2,27.79,toys,1450.0,68.0,3.0,48.0
4,Dd0QnrMk9Cj5,mVz5LO2Vd6cL,delivered,2017-12-22 16:44:04,2017-12-22 17:31:31,2018-01-05 19:22:49,2018-01-18,88868,ilhabela,SP,...,30.66,1,credit_card,1,76.15,toys,300.0,17.0,4.0,12.0


In [ ]:
ori_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89316 entries, 0 to 89315
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       89316 non-null  object        
 1   customer_id                    89316 non-null  object        
 2   order_status                   89316 non-null  object        
 3   order_purchase_timestamp       89316 non-null  datetime64[ns]
 4   order_approved_at              89307 non-null  datetime64[ns]
 5   order_delivered_timestamp      87427 non-null  datetime64[ns]
 6   order_estimated_delivery_date  89316 non-null  datetime64[ns]
 7   customer_zip_code_prefix       89316 non-null  int64         
 8   customer_city                  89316 non-null  object        
 9   customer_state                 89316 non-null  object        
 10  product_id                     89316 non-null  object        
 11  seller_id      

In [ ]:
ori_train_df.loc[ori_train_df.order_delivered_timestamp.isnull(), ["order_status", "order_delivered_timestamp"]]

,order_status,order_delivered_timestamp
15,canceled,NaT
41,shipped,NaT
86,shipped,NaT
89,canceled,NaT
133,canceled,NaT
...,...,...
89196,shipped,NaT
89226,canceled,NaT
89234,shipped,NaT
89262,canceled,NaT


## Labelling

In [ ]:
train_df = ori_train_df.copy()
print("Train shape before drop", train_df.shape)
train_df.dropna(subset=["order_delivered_timestamp"], inplace=True)
print("Train shape after drop", train_df.shape)

Train shape before drop (89316, 23)
Train shape after drop (87427, 23)


In [ ]:
train_df["is_late"] = (train_df["order_delivered_timestamp"] > train_df["order_estimated_delivery_date"]).astype(int)
train_df["is_late"].value_counts()

0    80689
1     6738
Name: is_late, dtype: int64

In [ ]:
train_df.to_csv("dataset/train_df.csv", index=False)
ori_test_df.to_csv("dataset/test_df.csv", index=False)

## EDA

In [ ]:
train_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,customer_state,...,payment_sequential,payment_type,payment_installments,payment_value,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,is_late
0,Axfy13Hk4PIk,hCT0x9JiGXBQ,delivered,2017-10-22 18:57:54,2017-10-22 19:14:13,2017-10-26 22:19:52,2017-11-09,58125,varzea paulista,SP,...,1,credit_card,1,259.14,toys,491.0,19.0,12.0,16.0,0
1,v6px92oS8cLG,PxA7fv9spyhx,delivered,2018-06-20 21:40:31,2018-06-20 22:20:20,2018-07-03 22:51:22,2018-07-24,3112,armacao dos buzios,RJ,...,1,credit_card,8,382.39,watches_gifts,440.0,18.0,14.0,17.0,0
2,Ulpf9skrhjfm,g3nXeJkGI0Qw,delivered,2018-02-16 16:19:31,2018-02-17 16:15:35,2018-02-27 01:29:50,2018-03-08,4119,jandira,SP,...,1,credit_card,4,249.25,costruction_tools_garden,2200.0,16.0,16.0,16.0,0
3,bwJVWupf2keN,EOEsCQ6QlpIg,delivered,2018-08-18 18:04:29,2018-08-18 18:15:16,2018-08-27 20:03:51,2018-09-19,18212,uberlandia,MG,...,1,credit_card,2,27.79,toys,1450.0,68.0,3.0,48.0,0
4,Dd0QnrMk9Cj5,mVz5LO2Vd6cL,delivered,2017-12-22 16:44:04,2017-12-22 17:31:31,2018-01-05 19:22:49,2018-01-18,88868,ilhabela,SP,...,1,credit_card,1,76.15,toys,300.0,17.0,4.0,12.0,0


In [ ]:
train_df.loc[train_df.order_approved_at.isnull(), "is_late"].value_counts()

0    9
Name: is_late, dtype: int64

In [ ]:
train_df.order_status.value_counts()

delivered    87422
canceled         5
Name: order_status, dtype: int64

In [ ]:
train_df.loc[train_df.order_status == "canceled"]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,customer_state,...,payment_sequential,payment_type,payment_installments,payment_value,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,is_late
14190,prmhq2USuP5I,svloFqXbk2Im,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-11-09 14:53:50,2016-12-08,26051,florianopolis,SC,...,1,credit_card,3,32.74,fashion_bags_accessories,2250.0,22.0,15.0,20.0,0
43014,b34KqtVskWuf,fvW8GM7pZATn,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-19 18:47:43,2016-11-30,74415,rio de janeiro,RJ,...,1,credit_card,5,161.01,toys,2100.0,22.0,50.0,32.0,0
45856,STGODjAZ1w0n,SOg6P6UXk8Nu,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-03-21 22:03:51,2018-03-09,35304,contagem,MG,...,1,credit_card,2,5.50,health_beauty,440.0,20.0,11.0,17.0,1
64316,gaNTtRHGtguq,Dq969FRq9Y67,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-16 14:36:59,2016-11-30,28140,sao paulo,SP,...,1,credit_card,2,106.78,health_beauty,1029.0,20.0,15.0,17.0,0
77637,F0WYhc9fTR0C,G0qHvWWfuMUu,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-14 15:07:11,2016-11-29,74415,rio de janeiro,RJ,...,1,credit_card,2,370.29,perfumery,610.0,22.0,14.0,16.0,0


In [ ]:
train_df.payment_sequential.value_counts()

1     83602
2      2534
3       486
4       232
5       150
6        98
7        78
8        44
9        34
11       33
10       31
12       23
13       14
15       11
14        9
18        8
17        8
19        8
21        7
20        6
16        5
22        2
23        1
24        1
26        1
25        1
Name: payment_sequential, dtype: int64

In [ ]:
# compare seller_id with is_late with mean and count of is_late and count of seller_id
train_df.groupby("seller_id").agg(
    {"is_late": ["mean", "sum"],
     "seller_id": "count"}
).sort_values(by=("is_late", "mean"), ascending=False)

is_late     seller_id
                mean sum     count
seller_id                         
Tk3rc1RErTUR     1.0   1         1
tjQoWRViCft1     1.0   2         2
4RCDmhw2Wb3Z     1.0   1         1
kBxE14RKcdCP     1.0   1         1
kbgVMiGW2l6E     1.0   1         1
...              ...  ..       ...
QQiQuKnrVLTq     0.0   0         8
QQ3QIAGssBWS     0.0   0         2
QL8dwGX89Ml9     0.0   0        12
QKUyOlEwWI1l     0.0   0         1
zyhKQqd3kGqW     0.0   0         4

[2822 rows x 3 columns]

In [ ]:
train_df.groupby("product_id").agg(
    {"is_late": ["mean", "sum"],
     "product_id": "count"}
).sort_values(by=("is_late", "mean"), ascending=False)

is_late     product_id
                mean sum      count
product_id                         
OTEDe61qFAL2     1.0   1          1
UoC4j9dEWiCw     1.0   2          2
2K7dRcj0is3k     1.0   1          1
REgcTjDv61D1     1.0   3          3
Z73pKMuHZ5DH     1.0   1          1
...              ...  ..        ...
MW4R3uc2ZQgG     0.0   0          2
MVzvxLOXlwH0     0.0   0          4
MVmjkrKxQROz     0.0   0          1
MVUOs7OvBaiQ     0.0   0          3
zztLAWHNx6cP     0.0   0          1

[26851 rows x 3 columns]

In [ ]:
train_df.groupby("customer_state").agg(
    {"is_late": ["mean", "sum"],
     "customer_state": "count"}
).sort_values(by=("is_late", "mean"), ascending=False)

is_late       customer_state
                    mean   sum          count
customer_state                               
AL              0.243077    79            325
MA              0.207297   125            603
PI              0.163814    67            409
SE              0.158249    47            297
CE              0.151680   167           1101
BA              0.139681   421           3014
TO              0.127572    31            243
RJ              0.126922  1428          11251
PA              0.116395    93            799
ES              0.115926   206           1777
PB              0.112500    54            480
RN              0.108753    41            377
RR              0.108108     4             37
PE              0.104498   151           1445
MS              0.099842    63            631
SC              0.093325   295           3161
GO              0.078194   142           1816
DF              0.067236   118           1755
RS              0.066131   321           4854
MT              0.061275    50            816
SP              0.055370  2054          37096
AC              0.054054     4             74
MG              0.053966   549          10173
PR              0.047405   211           4451
RO              0.046414    11            237
AP              0.029851     2             67
AM              0.028986     4            138

In [ ]:
train_df.groupby("customer_zip_code_prefix").agg(
    {"is_late": ["mean", "sum"],
     "customer_zip_code_prefix": "count"}
).sort_values(by=("is_late", "mean"), ascending=False)

is_late     customer_zip_code_prefix
                            mean sum                    count
customer_zip_code_prefix                                     
2803                         1.0   1                        1
97572                        1.0   1                        1
27516                        1.0   3                        3
37600                        1.0   1                        1
77817                        1.0   1                        1
...                          ...  ..                      ...
22080                        0.0   0                        4
22081                        0.0   0                        3
22210                        0.0   0                        6
22211                        0.0   0                        3
35365                        0.0   0                        3

[13842 rows x 3 columns]

In [ ]:
train_df.customer_city.value_counts()

sao paulo               14021
rio de janeiro           6064
belo horizonte           2380
brasilia                 1740
curitiba                 1405
                        ...  
alto taquari                1
tecainda                    1
santa luzia do norte        1
rodolfo fernandes           1
macuco de minas             1
Name: customer_city, Length: 3714, dtype: int64

In [ ]:
train_df.payment_type.value_counts()

credit_card    64435
wallet         16939
voucher         4790
debit_card      1263
Name: payment_type, dtype: int64

In [ ]:
pd.crosstab(train_df.payment_type, train_df.is_late)

is_late,0,1
payment_type,,
credit_card,59559,4876
debit_card,1157,106
voucher,4477,313
wallet,15496,1443


In [ ]:
train_df.groupby("is_late").price.mean()

is_late
0    341.138346
1    355.869184
Name: price, dtype: float64

In [ ]:
train_df.isnull().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                  9
order_delivered_timestamp          0
order_estimated_delivery_date      0
customer_zip_code_prefix           0
customer_city                      0
customer_state                     0
product_id                         0
seller_id                          0
price                              0
shipping_charges                   0
payment_sequential                 0
payment_type                       0
payment_installments               0
payment_value                      0
product_category_name            289
product_weight_g                  15
product_length_cm                 15
product_height_cm                 15
product_width_cm                  15
is_late                            0
dtype: int64

In [ ]:
cat_cols = [
    # "order_status",
    "customer_city",
    "customer_state",
    "payment_type",
    "product_category_name",
    "product_id",
    "seller_id"
]

## Baseline

In [ ]:
train_df.dropna(subset=["order_approved_at"], inplace=True)
train_df.loc[train_df.product_category_name.isnull(), "product_category_name"] = "unknown"
train_df.shape

(87418, 24)

In [ ]:
X = train_df.drop([
    "is_late",
    "order_id",
    "customer_id",
    "order_delivered_timestamp",
    "order_estimated_delivery_date",
    "order_status",
    "customer_zip_code_prefix",
], axis=1)
y = train_df["is_late"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((69934, 17), (17484, 17), (69934,), (17484,))

In [ ]:
cbc_model = cb.CatBoostClassifier(
    cat_features=cat_cols,
    # n_estimators=1000,
    # learning_rate=0.1,
    eval_metric='Accuracy',
    task_type='GPU',
    devices='0',
    early_stopping_rounds=200,
    verbose=100,
    random_seed=42,
)
cbc_model.fit(X_train, y_train, eval_set=(X_val, y_val))

Learning rate set to 0.049628
0:	learn: 0.9229273	test: 0.9229009	best: 0.9229009 (0)	total: 24.6ms	remaining: 24.6s
100:	learn: 0.9300626	test: 0.9331389	best: 0.9331961 (96)	total: 3.13s	remaining: 27.9s
200:	learn: 0.9307776	test: 0.9337680	best: 0.9338824 (194)	total: 6.52s	remaining: 25.9s
300:	learn: 0.9315640	test: 0.9344544	best: 0.9346831 (267)	total: 8.98s	remaining: 20.9s
400:	learn: 0.9322218	test: 0.9347975	best: 0.9348547 (392)	total: 11.4s	remaining: 17s
500:	learn: 0.9331370	test: 0.9350263	best: 0.9350263 (499)	total: 14.2s	remaining: 14.1s
600:	learn: 0.9338805	test: 0.9355983	best: 0.9357127 (592)	total: 19.1s	remaining: 12.7s
700:	learn: 0.9343381	test: 0.9354267	best: 0.9357127 (592)	total: 23.2s	remaining: 9.89s
bestTest = 0.9357126516
bestIteration = 592
Shrink model to first 593 iterations.


In [ ]:
y_val_pred = cbc_model.predict(X_val)
print(accuracy_score(y_val, y_val_pred))

0.9357126515671471


In [ ]:
cbc_model_full = cb.CatBoostClassifier(
    cat_features=cat_cols,
    # n_estimators=1000,
    # learning_rate=0.1,
    eval_metric='Accuracy',
    task_type='GPU',
    devices='0',
    early_stopping_rounds=200,
    verbose=100,
    random_seed=42,
)
cbc_model_full.fit(X, y)

Learning rate set to 0.026362
0:	learn: 0.9229221	total: 22.8ms	remaining: 22.7s
100:	learn: 0.9291221	total: 3.7s	remaining: 32.9s
200:	learn: 0.9309639	total: 7.05s	remaining: 28s
300:	learn: 0.9315587	total: 10.1s	remaining: 23.5s
400:	learn: 0.9320049	total: 12.9s	remaining: 19.2s
500:	learn: 0.9323137	total: 16.5s	remaining: 16.4s
600:	learn: 0.9325654	total: 20.2s	remaining: 13.4s
700:	learn: 0.9328399	total: 24s	remaining: 10.2s
800:	learn: 0.9330801	total: 29.3s	remaining: 7.29s
900:	learn: 0.9333890	total: 32s	remaining: 3.52s
999:	learn: 0.9336064	total: 35s	remaining: 0us


In [ ]:
test_df = ori_test_df.copy()
test_df = test_df.loc[:, X.columns.tolist()]
test_df.loc[test_df.product_category_name.isnull(), "product_category_name"] = "unknown"
test_df.head()

,order_purchase_timestamp,order_approved_at,customer_city,customer_state,product_id,seller_id,price,shipping_charges,payment_sequential,payment_type,payment_installments,payment_value,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,2017-11-18 12:29:57,2017-11-18 12:46:08,goiania,GO,1slxdgbgWFax,3jwvL6ihC45G,24.10,20.90,1,credit_card,2,155.77,toys,50.0,16.0,5.0,11.0
1,2018-06-02 17:13:12,2018-06-02 20:12:23,viamao,RS,77PgsiElQLeB,GlLj704QXlDB,42.89,12.28,1,credit_card,1,4.07,electronics,200.0,21.0,7.0,14.0
2,2018-01-08 11:01:30,2018-01-09 07:24:03,campinas,SP,QVlD26X1y7NI,V3iKL8r9W9NR,50.21,67.11,1,wallet,1,381.59,furniture_decor,1000.0,100.0,5.0,20.0
3,2017-03-10 10:24:46,2017-03-10 10:24:46,santana de parnaiba,SP,yWlFGkKYfrpa,RNBdBKsXebna,89.10,62.05,1,credit_card,3,14.76,toys,8950.0,40.0,30.0,40.0
4,2017-12-02 10:04:07,2017-12-05 04:13:30,aripuana,MT,h6MCbrwh5kiC,5Ja2lH0N2OZt,2139.99,9.41,1,wallet,1,284.09,toys,2301.0,32.0,35.0,34.0


In [ ]:
test_df.isnull().sum()

order_purchase_timestamp     0
order_approved_at            7
customer_city                0
customer_state               0
product_id                   0
seller_id                    0
price                        0
shipping_charges             0
payment_sequential           0
payment_type                 0
payment_installments         0
payment_value                0
product_category_name        0
product_weight_g            10
product_length_cm           10
product_height_cm           10
product_width_cm            10
dtype: int64

In [ ]:
test_df.loc[test_df.order_approved_at.isnull(), "order_approved_at"].index

Int64Index([309, 4699, 16372, 21613, 28627, 30628, 36653], dtype='int64')

In [ ]:
test_df.loc[test_df.order_approved_at.isnull(), "order_approved_at"] = \
    test_df.loc[test_df.order_approved_at.isnull(), "order_purchase_timestamp"] + \
    (test_df.order_approved_at - test_df.order_purchase_timestamp).mean()

In [ ]:
test_df.loc[[309, 4699, 16372, 21613, 28627, 30628, 36653], :]

,order_purchase_timestamp,order_approved_at,customer_city,customer_state,product_id,seller_id,price,shipping_charges,payment_sequential,payment_type,payment_installments,payment_value,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
309,2017-02-18 22:49:19,2017-02-19 09:17:45.739940426,rio de janeiro,RJ,CY8OZ3lT9uqB,sevytuBWdSFl,2899.00,2.46,1,wallet,1,106.56,telephony,250.0,18.0,3.0,28.0
4699,2017-02-18 12:45:31,2017-02-18 23:13:57.739940426,novo hamburgo,RS,q5EIWgf7mpli,CtMaPBruHzFA,31.58,54.64,1,wallet,1,152.21,toys,1008.0,33.0,14.0,26.0
16372,2017-02-18 22:49:19,2017-02-19 09:17:45.739940426,rio de janeiro,RJ,CY8OZ3lT9uqB,sevytuBWdSFl,2899.00,2.46,1,wallet,1,106.56,telephony,250.0,18.0,3.0,28.0
21613,2017-02-19 01:28:47,2017-02-19 11:57:13.739940426,sao paulo,SP,KsJ0qytYsZlU,2a07LobkzHUx,434.99,83.60,1,wallet,1,93.52,toys,1700.0,33.0,8.0,23.0
28627,2017-01-19 12:48:08,2017-01-19 23:16:34.739940426,santos,SP,qJJWuXhXxynp,3cqSxyYyqpx6,475.00,2.46,1,wallet,1,67.08,toys,700.0,26.0,16.0,21.0
30628,2017-02-18 17:15:03,2017-02-19 03:43:29.739940426,sao jose de ribamar,MA,6Ww1i7qd1Zqu,YVBAT3QqF9Qj,58.10,67.33,1,wallet,1,258.08,cool_stuff,1200.0,42.0,25.0,15.0
36653,2017-02-18 13:29:47,2017-02-18 23:58:13.739940426,sao paulo,SP,3CttGRrxTrK5,u3MhLPomw5UN,111.27,75.01,1,wallet,1,172.33,toys,3100.0,28.0,28.0,50.0


In [ ]:
y_test_pred = cbc_model_full.predict(test_df)
y_test_pred[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64)

In [ ]:
submission_df = pd.DataFrame({
    "order_id": ori_test_df.order_id,
    "is_late": y_test_pred
})
submission_df.head()

,order_id,is_late
0,u6rPMRAYIGig,0
1,ohY8f4FEbX19,0
2,I28liQek73i2,0
3,bBG1T89mlY8W,0
4,CYxJJSQS8Lbo,0


In [ ]:
submission_df.is_late.value_counts()

0    37445
1      834
Name: is_late, dtype: int64

In [ ]:
submission_df.to_csv("submissions/submission_baseline.csv", index=False)